### **Sistema Híbrido de Predicción y Gestión de Riesgo (FNN + GARCH)**

https://es.finance.yahoo.com/quote/RB%3DF/

+ Clara Aguilar 

+ Samantha Sanchez 

+ Juan Pablo Colome

+ Isabel Valladolid

In [53]:
%%capture
# !pip install  scikit-learn tensorflow
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import acf, pacf
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

In [54]:
# Descargar datos
df = yf.download("RB=F", period="10y", interval="1d")

df.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,RB=F,RB=F,RB=F,RB=F,RB=F
Date,,,,,
2016-04-12,1.5343,1.5386,1.4932,1.5023,58656
2016-04-13,1.5295,1.5384,1.5059,1.5277,62404
2016-04-14,1.5056,1.5425,1.4957,1.5273,46239
2016-04-15,1.4612,1.5126,1.4534,1.5064,50578
2016-04-18,1.4365,1.4617,1.3990,1.4229,49114


In [55]:
df = df.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume"
})

df = df.dropna()

In [56]:
# cnvertir el índice a datetime y ordenar por fecha
df.index = pd.to_datetime(df.index)
df = df.sort_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df[("close", "RB=F")],
    mode="lines",
    name="Precio"
))

fig.update_layout(
    title="Precio de RBOB Gasoline (RB=F)",
    xaxis_title="Fecha",
    yaxis_title="Precio",
    template="plotly_white",
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show()

## Modelo FFNN

In [57]:
# Usamos solo el precio de cierre
serie_tiempo = df['close'].dropna().values.reshape(-1, 1)

In [58]:
# Seleccionamos train y test (80/20)
train_size = int(len(serie_tiempo) * 0.8)
train_data = serie_tiempo[:train_size]
test_data = serie_tiempo[train_size:]

In [59]:
# Iniciamos el escalador
scaler = MinMaxScaler(feature_range=(0, 1))

# Aplicamos el escalador SOLO a los datos de entrenamiento
train_scaled = scaler.fit_transform(train_data)

# Ahora escalador con parametros del train, pero para transformar los datos de test
test_scaled = scaler.transform(test_data)

In [60]:
# Función para crear ventanas deslizantes
def crear_ventanas(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size), 0])
        y.append(data[i + window_size, 0])
    return np.array(X), np.array(y)

window_size = 10
X_train, y_train = crear_ventanas(train_scaled, window_size)
X_test, y_test = crear_ventanas(test_scaled, window_size)

In [61]:
# Iniciamos el modelo FFNN
model = Sequential([
    # Capa 1: input_dim DEBE coincidir con el window_size
    Dense(16, activation='relu', input_dim=window_size, name='Capa_Oculta_1'),

    # Capa 2: Extracción de características no lineales
    Dense(8, activation='relu', name='Capa_Oculta_2'),

    # Capa de Salida: 1 neurona Funcion de act LINEAL
    Dense(1, name='Salida_Pronostico')
])
model.summary()

# Usamos el optimizador Adam y MSE
optimizador = Adam(learning_rate=0.005)
model.compile(optimizer=optimizador, loss='mse')

history = model.fit(
    X_train, y_train,
    epochs=60,           # Cantidad de veces que verá el dataset completo
    batch_size=8,        # Actualiza los pesos cada 8 ventanas
    validation_data=(X_test, y_test), # Evaluamos en datos que no ha visto
    verbose=0            # 0 para no mostrar info
)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Capa_Oculta_1 (Dense)           │ (None, 16)             │           176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Capa_Oculta_2 (Dense)           │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Salida_Pronostico (Dense)       │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [62]:
# Predicciones
preds = model.predict(X_test)

# Asegurar formato 2D
preds = preds.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

# Desescalar
preds = scaler.inverse_transform(preds)
y_test_real = scaler.inverse_transform(y_test)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


In [63]:
# Copia del dataframe
df_plot = df.copy()

# Asegurar índice datetime y ordenado
df_plot.index = pd.to_datetime(df_plot.index)
df_plot = df_plot.sort_index()

# Tomar la columna close real desde el MultiIndex
df_plot["real"] = df_plot[("close", "RB=F")]

# Crear columna de predicción
df_plot["pred"] = np.nan

# Insertar predicciones en el tramo final
df_plot.iloc[-len(preds):, df_plot.columns.get_loc("pred")] = preds.flatten()

# Crear gráfica interactiva
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot.index,
    y=df_plot["real"],
    mode="lines",
    name="Real"
))

fig.add_trace(go.Scatter(
    x=df_plot.index,
    y=df_plot["pred"],
    mode="lines",
    name="Predicción"
))

fig.update_layout(
    title="Precio Real vs Predicción - RBOB",
    xaxis_title="Fecha",
    yaxis_title="Precio",
    template="plotly_white",
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show()

In [64]:
# Métricas
mae = mean_absolute_error(y_test_real, preds)
rmse = np.sqrt(mean_squared_error(y_test_real, preds))

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")

MAE: 0.040
RMSE: 0.059


In [65]:
real = np.sign(np.diff(y_test_real.flatten()))
pred = np.sign(np.diff(preds.flatten()))

DA = np.mean(real == pred)

print("Directional Accuracy:", DA)

Directional Accuracy: 0.483739837398374


#### Segundo modelo fnn

In [66]:
# Función para crear ventanas deslizantes
def crear_ventanas(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size), 0])
        y.append(data[i + window_size, 0])
    return np.array(X), np.array(y)

window_size = 20
X_train, y_train = crear_ventanas(train_scaled, window_size)
X_test, y_test = crear_ventanas(test_scaled, window_size)

In [67]:
model_two = Sequential([
   Dense(128, activation='relu', input_shape=(window_size,)),
   Dropout(0.2),
   Dense(64, activation='relu'),
   Dropout(0.2),
   Dense(32, activation='relu'),
   Dropout(0.1),
   Dense(1, activation='sigmoid')
])

model_two.summary()

optimizador = Adam(learning_rate=0.001)
model_two.compile(optimizer=optimizador, loss='mse')

history = model_two.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0
)

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_29 (Dense)                │ (None, 128)            │         2,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,057 (51.00 KB)

 Trainable params: 13,057 (51.00 KB)

 Non-trainable params: 0 (0.00 B)

### Predicciones para el 13 de abril 

In [68]:
# Fechas objetivo: 13-17 Abril 2026 (Lunes a Viernes)
target_dates = pd.date_range(start='2026-04-13', periods=5, freq='B')
print(f"Fechas objetivo para predicción:")
for date in target_dates:
    print(f"  - {date.strftime('%Y-%m-%d')} ({date.strftime('%A')})")

# Usar últimos 20 días del dataset como entrada para predicción
last_sequence = train_scaled[-window_size:].reshape(1, window_size)
predictions_two = []
current_sequence = last_sequence.copy()

print(f"\nGenerando predicciones secuenciales")
for i in range(5):  # 5 días
    next_pred = model_two.predict(current_sequence, verbose=0)[0, 0]
    predictions_two.append(next_pred)
    
    # Actualizar secuencia
    current_sequence = np.roll(current_sequence, -1, axis=1)
    current_sequence[0, -1] = next_pred

# Desnormalizar predicciones
predictions_two_actual = scaler.inverse_transform(np.array(predictions_two).reshape(-1, 1)).flatten()

# Crear DataFrame de predicciones
df_forecast_two = pd.DataFrame({
    'date': target_dates,
    'day_of_week': target_dates.strftime('%A'),
    'predicted_close': predictions_two_actual
})

print("\n Predicciones 13-17 Abril")
print(df_forecast_two.to_string(index=False))

# Estadísticas de predicción
print(f"\nEstadísticas de Predicción:")
print(f"  Precio promedio predicho: ${predictions_two_actual.mean():.2f}")
print(f"  Mínimo predicho: ${predictions_two_actual.min():.2f}")
print(f"  Máximo predicho: ${predictions_two_actual.max():.2f}")
print(f"  Volatilidad (std): ${predictions_two_actual.std():.4f}")

Fechas objetivo para predicción:
  - 2026-04-13 (Monday)
  - 2026-04-14 (Tuesday)
  - 2026-04-15 (Wednesday)
  - 2026-04-16 (Thursday)
  - 2026-04-17 (Friday)

Generando predicciones secuenciales

 Predicciones 13-17 Abril
      date day_of_week  predicted_close
2026-04-13      Monday         2.677850
2026-04-14     Tuesday         2.633045
2026-04-15   Wednesday         2.598477
2026-04-16    Thursday         2.565287
2026-04-17      Friday         2.532828

Estadísticas de Predicción:
  Precio promedio predicho: $2.60
  Mínimo predicho: $2.53
  Máximo predicho: $2.68
  Volatilidad (std): $0.0507


In [72]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df[("close", "RB=F")],
    mode="lines",
    name="Precio histórico"
))

fig.add_trace(go.Scatter(
    x=df_forecast_two["date"],
    y=df_forecast_two["predicted_close"],
    mode="lines+markers",
    name="Predicción 13-17 Abr",
    line=dict(color="red", width=2, dash="dash")
))

fig.update_layout(
    title="Predicción de cierre para 13-17 Abril 2026",
    xaxis_title="Fecha",
    yaxis_title="Precio",
    template="plotly_white",
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show()

### Modelo Garch

In [70]:
# Calcular retornos
returns = 100 * np.log(df['close'] / df['close'].shift(1)).dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines', name='Retornos'))
fig.update_layout(title=f'Retornos Diarios',
                  yaxis_title='Retornos (%)')
fig.show()

In [71]:
# Retornos al cuadrado
sq_returns = returns**2

# Calcular ACF y PACF
lag_acf = acf(sq_returns, nlags=20)
lag_pacf = pacf(sq_returns, nlags=20, method='ols')

# Graficar ACF y PACF con Plotly
fig = make_subplots(rows=1, cols=2, subplot_titles=('ACF de Retornos al Cuadrado', 'PACF de Retornos al Cuadrado'))

# Añadir barras de ACF
fig.add_trace(go.Bar(x=np.arange(len(lag_acf)), y=lag_acf, name='ACF'), row=1, col=1)
# Añadir barras de PACF
fig.add_trace(go.Bar(x=np.arange(len(lag_pacf)), y=lag_pacf, name='PACF'), row=1, col=2)

# Líneas de significancia (aprox 1.96 / sqrt(N))
sig_level = 1.96 / np.sqrt(len(returns))
for i in [1, 2]:
    fig.add_hline(y=sig_level, line_dash="dash", line_color="red", row=1, col=i)
    fig.add_hline(y=-sig_level, line_dash="dash", line_color="red", row=1, col=i)

fig.update_layout(title='Análisis de Dependencia de Varianza', showlegend=False)
fig.show()

In [73]:
# Ajuste del modelo GARCH(1,1)
am = arch_model(returns, vol='Garch', p=1, q=1, dist='t')
res = am.fit(disp='off')


# Extraemos la volatilidad condicional modelada (Varianza que predijimos)
cond_vol = res.conditional_volatility

# Cálculo del Value at Risk (VaR) Histórico Condicional (GARCH) al 5%
# Cuantil empírico al 5% de los residuales estandarizados
q_5 = res.std_resid.quantile(0.05)

# VaR = Media Condicional + (Cuantil * Volatilidad Condicional)
# GARCH por defecto asume media constante, la extraemos de los parámetros
mean = res.params['mu']
VaR_5 = mean + cond_vol * q_5

# 5. Graficar los Retornos vs el VaR predictivo
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines',
                         name='Retornos Reales', opacity=0.6))
fig.add_trace(go.Scatter(x=VaR_5.index, y=VaR_5, mode='lines',
                         name='VaR 5% (GARCH)', line=dict(color='red')))

fig.update_layout(title='Backtesting de Riesgo: Retornos de XOM vs GARCH(1,1) VaR al 5%',
                  yaxis_title='Retornos (%)')
fig.show()

In [74]:
print(res.summary())

                        Constant Mean - GARCH Model Results                         
Dep. Variable:                         RB=F   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -5560.27
Distribution:      Standardized Student's t   AIC:                           11130.5
Method:                  Maximum Likelihood   BIC:                           11159.7
                                              No. Observations:                 2514
Date:                      Sun, Apr 12 2026   Df Residuals:                     2513
Time:                              00:06:27   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu        